# Glove Data Visualiser — Full-Hand PDF Export

Visualise raw and filtered glove sensor data **for both hands and every segment** in a single landscape PDF.

**Layout:**
- **Section 0** — Install / import dependencies
- **Section 1** — Configuration (data path, sampling rate, output path)
- **Section 2** — Load a single CSV trial
- **Section 3** — Filter configuration (choose technique and parameters)
- **Section 4** — Generate the multi-page landscape PDF

---
**Column naming conventions in the dataset:**
```
Fingers : {hand}_{finger}_{loc}_{channel}      e.g. left_thumb_mid_ax,  left_thumb_pip_flex
Palm    : {hand}_palm_mid_{channel}            e.g. left_palm_mid_pitch (single IMU, no flex)
Wrist   : {hand}_wrist_{channel}               e.g. left_wrist_heading  (single IMU, no flex)
```

**PDF layout:**
- One landscape page per signal-group row, with one column per segment.
- Columns ordered: **L Wrist · L Palm · L Thumb · L Index · L Middle · L Ring · L Pinky · R Wrist · R Palm · R Thumb · R Index · R Middle · R Ring · R Pinky**.
- Rows: Mid Accel · Mid Orientation · Prox Accel · Prox Orientation · Flex sensors.
- Wrist/palm columns omit the prox and flex rows (cells left blank with a "n/a" note).

---
## 0. Dependencies

In [24]:
import subprocess, sys
pkgs = ['pandas', 'numpy', 'matplotlib', 'scipy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('Dependencies ready.')

Dependencies ready.


In [25]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import MaxNLocator, AutoMinorLocator
from scipy import signal as sp_signal

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Imports OK.')

Imports OK.


---
## 1. Configuration

Edit this cell to control which file is visualised and where the PDF is written.

In [26]:
# =============================================================================
# 1A.  DATA PATH
# =============================================================================
# Path to a gesture FOLDER (one of the label subdirectories) OR a single CSV.
# If a folder is given, the first CSV inside is loaded by default.
DATA_PATH = '/home/jestin/ThesisRepo/ML/NewTestData/8_Jestin/Dynamic/Double_Pistol_Recoil/glove_data_L_Dynamic_Double_Pistol_Recoil_3s_2_2026-05-05_07-33-50.csv'

# If DATA_PATH is a folder, which file index to pick (0 = first file)
FILE_INDEX = 0

# =============================================================================
# 1B.  SAMPLING RATE
# =============================================================================
# Approximate sample rate of the glove in Hz (used for time-axis scaling and filter design).
FS_HZ = 30.0


# =============================================================================
# 1C.  TIME AXIS
# =============================================================================
# If True, use a *_time_ms column in the CSV as the time axis (ms -> s).
# If False, a synthetic time axis is generated from FS_HZ.
USE_TIMESTAMP_COLUMN = False
TIMESTAMP_COL        = None    # e.g. 'left_recv_time_ms'.  Leave None to auto-detect.

# =============================================================================
# 1D.  PDF OUTPUT
# =============================================================================
# Where to write the PDF. Leave None to auto-generate next to the source CSV.
# PDF_OUTPUT_PATH = f"/home/jestin/ThesisRepo/ML/NewReports/DataGraphs/glove_data_{DATA_PATH.split('/')[6]}_{os.path.basename(DATA_PATH).split('_')[4:7]}_visualization_butterworth4_lp_10hz_raw.pdf"


# # Per-page figure size (inches). A3 landscape ~ (16.5, 11.7). Use bigger if cramped.
# PDF_FIG_SIZE = (44, 24)

# print('Configuration loaded.')
# print(f'  Sampling rate: {FS_HZ} Hz')
# print(f'  Data path    : {DATA_PATH}')

---
## 2. Load Data

In [27]:
# Resolve file path -----------------------------------------------------------
data_path = os.path.expanduser(DATA_PATH)

if os.path.isdir(data_path):
    csv_files = sorted(glob.glob(os.path.join(data_path, '**/*.csv'), recursive=True))
    if not csv_files:
        raise FileNotFoundError(f'No CSV files found under: {data_path}')
    csv_path = csv_files[FILE_INDEX]
    print(f'Found {len(csv_files)} CSV(s) in folder. Loading index {FILE_INDEX}:')
    print(f'  {os.path.basename(csv_path)}')
elif os.path.isfile(data_path):
    csv_path = data_path
    print(f'Loading file: {os.path.basename(csv_path)}')
else:
    raise FileNotFoundError(
        f"DATA_PATH not found: '{data_path}'\n"
        "Please update DATA_PATH in Section 1 to point to your data folder or CSV."
    )

# Read CSV --------------------------------------------------------------------
df_raw = pd.read_csv(csv_path)
print(f'\nLoaded: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')

# Build time axis -------------------------------------------------------------
if USE_TIMESTAMP_COLUMN:
    ts_col = TIMESTAMP_COL
    if ts_col is None:
        candidates = [c for c in df_raw.columns if 'time_ms' in c]
        ts_col = candidates[0] if candidates else None
    if ts_col and ts_col in df_raw.columns:
        t = (df_raw[ts_col].values - df_raw[ts_col].values[0]) / 1000.0  # ms -> s
        print(f'Time axis from column: {ts_col}  (duration: {t[-1]:.2f} s)')
    else:
        t = np.arange(len(df_raw)) / FS_HZ
        print(f'Synthetic time axis at {FS_HZ} Hz  (duration: {t[-1]:.2f} s)')
else:
    t = np.arange(len(df_raw)) / FS_HZ
    print(f'Synthetic time axis at {FS_HZ} Hz  (duration: {t[-1]:.2f} s)')

# Helper: safe column getter --------------------------------------------------
def get_col(df, name):
    """Return column as numpy array, or NaN array if missing/non-numeric."""
    if name in df.columns:
        s = pd.to_numeric(df[name], errors='coerce')
        return s.to_numpy(dtype=float)
    return np.full(len(df), np.nan)

print('\nRaw data loaded. Proceed to Section 3 to configure filtering.')

Loading file: glove_data_L_Dynamic_Double_Pistol_Recoil_3s_2_2026-05-05_07-33-50.csv

Loaded: 92 rows x 295 columns
Synthetic time axis at 30.0 Hz  (duration: 3.03 s)

Raw data loaded. Proceed to Section 3 to configure filtering.


---
## 3. Filter Configuration

Choose a filtering technique and tune its parameters here. Run **Section 4** after changing anything to regenerate the PDF.

In [28]:
# =============================================================================
# SELECT FILTER
# =============================================================================
# Options:
#   'none'              -- no filtering, pass raw signal through
#   'butterworth_lp'    -- Butterworth low-pass (removes high-freq noise)
#   'butterworth_hp'    -- Butterworth high-pass (removes DC / slow drift)
#   'butterworth_bp'    -- Butterworth band-pass
#   'moving_average'    -- Simple moving average (causal)
#   'savgol'            -- Savitzky-Golay (smooths while preserving peaks)
#   'median'            -- Median filter (good for spike / impulse noise)

FILTER_TYPE = 'butterworth_lp'

# Butterworth parameters (used for butterworth_lp / hp / bp)
BW_ORDER     = 4
BW_CUTOFF_LO = 3.0
BW_CUTOFF_HI = 30.0
BW_ZERO_PHASE = True

# Moving average parameters
MA_WINDOW = 5

# Savitzky-Golay parameters
SG_WINDOW = 11
SG_POLYORDER = 3

# Median filter parameters
MED_KERNEL = 5

# =============================================================================
# Z-SCORE NORMALISATION
# =============================================================================
# When True, each plotted signal is standardised to zero mean and unit
# variance (per-channel, per-trial) AFTER filtering.  Useful for comparing
# the shape of a trial across sensors/segments on a common scale.
#   z = (x - mean(x)) / std(x)
# Channels with zero variance (constant signals) are left at zero.
USE_ZSCORE = False

def zscore(sig):
    """Return the z-score of a 1-D signal. NaN-safe; constant signals -> zeros."""
    sig = np.asarray(sig, dtype=float)
    if sig.size == 0 or np.isnan(sig).all():
        return sig
    mu = np.nanmean(sig)
    sd = np.nanstd(sig)
    if not np.isfinite(sd) or sd == 0.0:
        return np.zeros_like(sig)
    return (sig - mu) / sd

# Apply filter function -------------------------------------------------------
def apply_filter(signal, filter_type, fs):
    """Apply the configured filter to a 1-D signal array. Returns filtered array."""
    sig = np.array(signal, dtype=float)
    if np.isnan(sig).all():
        return sig
    # Replace NaNs with column mean so filtering is stable
    if np.isnan(sig).any():
        m = np.nanmean(sig)
        sig = np.where(np.isnan(sig), m if np.isfinite(m) else 0.0, sig)

    ft = filter_type.lower()
    nyq = fs / 2.0

    if ft == 'none':
        return sig
    if ft == 'butterworth_lp':
        cutoff = min(BW_CUTOFF_LO / nyq, 0.999)
        b, a = sp_signal.butter(BW_ORDER, cutoff, btype='low')
        return (sp_signal.filtfilt if BW_ZERO_PHASE else sp_signal.lfilter)(b, a, sig)
    if ft == 'butterworth_hp':
        cutoff = min(BW_CUTOFF_LO / nyq, 0.999)
        b, a = sp_signal.butter(BW_ORDER, cutoff, btype='high')
        return (sp_signal.filtfilt if BW_ZERO_PHASE else sp_signal.lfilter)(b, a, sig)
    if ft == 'butterworth_bp':
        lo = min(BW_CUTOFF_LO / nyq, 0.499)
        hi = min(BW_CUTOFF_HI / nyq, 0.999)
        b, a = sp_signal.butter(BW_ORDER, [lo, hi], btype='band')
        return (sp_signal.filtfilt if BW_ZERO_PHASE else sp_signal.lfilter)(b, a, sig)
    if ft == 'moving_average':
        kernel = np.ones(MA_WINDOW) / MA_WINDOW
        return np.convolve(sig, kernel, mode='same')
    if ft == 'savgol':
        win = SG_WINDOW if SG_WINDOW % 2 == 1 else SG_WINDOW + 1
        return sp_signal.savgol_filter(sig, win, SG_POLYORDER)
    if ft == 'median':
        ker = MED_KERNEL if MED_KERNEL % 2 == 1 else MED_KERNEL + 1
        return sp_signal.medfilt(sig, ker)
    raise ValueError(f'Unknown FILTER_TYPE: "{filter_type}"')


def filter_description():
    ft = FILTER_TYPE.lower()
    if ft == 'none':             return 'No filter'
    if ft == 'butterworth_lp':   return f'Butterworth LP  (order={BW_ORDER}, fc={BW_CUTOFF_LO} Hz, {"zero-phase" if BW_ZERO_PHASE else "causal"})'
    if ft == 'butterworth_hp':   return f'Butterworth HP  (order={BW_ORDER}, fc={BW_CUTOFF_LO} Hz, {"zero-phase" if BW_ZERO_PHASE else "causal"})'
    if ft == 'butterworth_bp':   return f'Butterworth BP  (order={BW_ORDER}, {BW_CUTOFF_LO}-{BW_CUTOFF_HI} Hz, {"zero-phase" if BW_ZERO_PHASE else "causal"})'
    if ft == 'moving_average':   return f'Moving Average  (window={MA_WINDOW} samples)'
    if ft == 'savgol':           return f'Savitzky-Golay  (window={SG_WINDOW}, poly={SG_POLYORDER})'
    if ft == 'median':           return f'Median Filter   (kernel={MED_KERNEL})'
    return FILTER_TYPE


print(f'Filter selected: {filter_description()}')
print(f'Z-score normalisation: {"ON" if USE_ZSCORE else "OFF"}')
print('Run Section 4 to build the PDF.')

Filter selected: Butterworth LP  (order=4, fc=3.0 Hz, zero-phase)
Z-score normalisation: OFF
Run Section 4 to build the PDF.


In [29]:
# =============================================================================
# 1D.  PDF OUTPUT
# =============================================================================
# Where to write the PDF. Leave None to auto-generate next to the source CSV.
PDF_OUTPUT_PATH = f"/home/jestin/ThesisRepo/ML/NewReports/DataGraphs/glove_data_{DATA_PATH.split('/')[6]}_{os.path.basename(DATA_PATH).split('_')[4:7]}_visualization_{filter_description()}_{ 'zscore' if USE_ZSCORE else 'raw'}.pdf"


# Per-page figure size (inches). A3 landscape ~ (16.5, 11.7). Use bigger if cramped.
PDF_FIG_SIZE = (44, 24)

print('Configuration loaded.')
print(f'  Sampling rate: {FS_HZ} Hz')
print(f'  Data path    : {DATA_PATH}')

Configuration loaded.
  Sampling rate: 30.0 Hz
  Data path    : /home/jestin/ThesisRepo/ML/NewTestData/8_Jestin/Dynamic/Double_Pistol_Recoil/glove_data_L_Dynamic_Double_Pistol_Recoil_3s_2_2026-05-05_07-33-50.csv


---
## 4. Build the multi-page landscape PDF

This generates a single landscape PDF where:
- **Each page** corresponds to one signal-group row (Mid Accel, Mid Orientation, Prox Accel, Prox Orientation, Flex sensors).
- **Each column** is one segment, in the order:
  `L Wrist | L Palm | L Thumb | L Index | L Middle | L Ring | L Pinky | R Wrist | R Palm | R Thumb | R Index | R Middle | R Ring | R Pinky`.
- The first page is a combined overview putting all rows on one giant page (one row per signal-group).

Wrist and palm have a single IMU and no flex sensors, so their cells in the prox/flex rows are marked **n/a**.

In [30]:
# Colours ---------------------------------------------------------------------
COLOURS = {
    # Accelerometer
    'ax':        '#E63946',
    'ay':        '#2A9D8F',
    'az':        '#457B9D',
    # Orientation
    'pitch':     '#F4A261',
    'roll':      '#264653',
    'yaw':       '#A8DADC',
    'heading':   '#A8DADC',  # wrist uses 'heading' instead of 'yaw'
    # Flex
    'mcp_flex':  '#6A0572',
    'pip_flex':  '#C77DFF',
}

# Column ordering -------------------------------------------------------------
# (hand, segment_kind, segment_name)
#   segment_kind: 'wrist' (single IMU, no flex/prox) | 'palm' (single IMU, no flex/prox) | 'finger'
SEGMENT_COLUMNS = []
for hand in ['left', 'right']:
    SEGMENT_COLUMNS.append((hand, 'wrist',  'wrist'))
    SEGMENT_COLUMNS.append((hand, 'palm',   'palm'))
    for finger in ['thumb', 'index', 'middle', 'ring', 'pinky']:
        SEGMENT_COLUMNS.append((hand, 'finger', finger))

# Row definitions: (row_label, plot_kind, loc)
#   plot_kind: 'accel' | 'ypr' | 'flex'
#   loc:       'mid' | 'prox' | None  (None for flex)
ROW_DEFS = [
    ('Mid IMU - Accelerometer',  'accel', 'mid'),
    ('Mid IMU - Orientation',    'ypr',   'mid'),
    ('Prox IMU - Accelerometer', 'accel', 'prox'),
    ('Prox IMU - Orientation',   'ypr',   'prox'),
    ('Flex sensors',             'flex',  None),
]

# Helpers ---------------------------------------------------------------------
def column_title(hand, kind, name):
    """Short header for the top of each column."""
    h = 'L' if hand == 'left' else 'R'
    return f'{h} {name.capitalize()}'


def plot_accel(ax, df, t, hand, kind, name, loc):
    """Draw ax/ay/az for the given segment into ax."""
    if kind == 'wrist':
        prefix = f'{hand}_wrist'                 # wrist has no loc
    elif kind == 'palm':
        if loc != 'mid':                         # palm: only mid IMU
            return False
        prefix = f'{hand}_palm_mid'
    else:
        prefix = f'{hand}_{name}_{loc}'

    plotted = False
    for ch in ['ax', 'ay', 'az']:
        col = f'{prefix}_{ch}'
        raw = get_col(df, col)
        if np.isnan(raw).all():
            continue
        filt = apply_filter(raw, FILTER_TYPE, FS_HZ)
        if USE_ZSCORE:
            filt = zscore(filt)
        ax.plot(t, filt, color=COLOURS[ch], linewidth=1.0, label=ch)
        plotted = True
    if plotted:
        ax.legend(loc='upper right', fontsize=6, framealpha=0.6, ncol=3,
                  handlelength=1.0, columnspacing=0.8)
    return plotted


def plot_ypr(ax, df, t, hand, kind, name, loc):
    """Draw yaw(or heading)/pitch/roll for the given segment into ax."""
    if kind == 'wrist':
        # Wrist channels: heading, pitch, roll
        prefix = f'{hand}_wrist'
        channels = [('heading', 'heading'), ('pitch', 'pitch'), ('roll', 'roll')]
    elif kind == 'palm':
        if loc != 'mid':
            return False
        prefix = f'{hand}_palm_mid'
        channels = [('yaw', 'yaw'), ('pitch', 'pitch'), ('roll', 'roll')]
    else:
        prefix = f'{hand}_{name}_{loc}'
        channels = [('yaw', 'yaw'), ('pitch', 'pitch'), ('roll', 'roll')]

    plotted = False
    for ch, label in channels:
        raw = get_col(df, f'{prefix}_{ch}')
        if np.isnan(raw).all():
            continue
        filt = apply_filter(raw, FILTER_TYPE, FS_HZ)
        if USE_ZSCORE:
            filt = zscore(filt)
        ax.plot(t, filt, color=COLOURS[ch], linewidth=1.0, label=label)
        plotted = True
    if plotted:
        ax.legend(loc='upper right', fontsize=6, framealpha=0.6, ncol=3,
                  handlelength=1.0, columnspacing=0.8)
    return plotted


def plot_flex(ax, df, t, hand, kind, name, loc):
    """Draw mcp_flex and pip_flex for the given finger into ax."""
    if kind != 'finger':
        return False
    plotted = False
    for ch in ['mcp_flex', 'pip_flex']:
        raw = get_col(df, f'{hand}_{name}_{ch}')
        if np.isnan(raw).all() or (raw == -1).all():
            continue
        filt = apply_filter(raw, FILTER_TYPE, FS_HZ)
        if USE_ZSCORE:
            filt = zscore(filt)
        ax.plot(t, filt, color=COLOURS[ch], linewidth=1.0, label=ch)
        plotted = True
    if plotted:
        ax.legend(loc='upper right', fontsize=6, framealpha=0.6, ncol=2,
                  handlelength=1.0, columnspacing=0.8)
    return plotted


def render_na(ax, msg='n/a'):
    ax.text(0.5, 0.5, msg, transform=ax.transAxes,
            ha='center', va='center', fontsize=9, color='#999')
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.grid(False)


def fill_subplot(ax, df, t, row_def, seg_def):
    """Render one (row, column) cell."""
    row_label, kind_plot, loc = row_def
    hand, seg_kind, seg_name  = seg_def

    # Cells that don't apply (wrist/palm in prox or flex rows)
    if (seg_kind in ('wrist', 'palm')) and (loc == 'prox' or kind_plot == 'flex'):
        render_na(ax)
        return

    if kind_plot == 'accel':
        ok = plot_accel(ax, df, t, hand, seg_kind, seg_name, loc)
    elif kind_plot == 'ypr':
        ok = plot_ypr(ax, df, t, hand, seg_kind, seg_name, loc)
    elif kind_plot == 'flex':
        ok = plot_flex(ax, df, t, hand, seg_kind, seg_name, loc)
    else:
        ok = False

    if not ok:
        render_na(ax, 'no data')
        return

    ax.tick_params(labelsize=6)



# ── Build PDF ────────────────────────────────────────────────────────────────
if PDF_OUTPUT_PATH is None:
    base, _ = os.path.splitext(csv_path)
    out_pdf = f'{base}_overview_{FILTER_TYPE}.pdf'
else:
    out_pdf = PDF_OUTPUT_PATH

ncols = len(SEGMENT_COLUMNS)
nrows_full = len(ROW_DEFS)

trial_name = os.path.basename(csv_path)
filt_desc  = filter_description()
scale_desc = 'z-score' if USE_ZSCORE else 'raw units'

with PdfPages(out_pdf) as pdf:
    # ---------- Page 1: combined overview (all rows on one giant page) -------
    fig = plt.figure(figsize=PDF_FIG_SIZE)
    gs = gridspec.GridSpec(
        nrows_full + 1, ncols,
        height_ratios=[0.18] + [1] * nrows_full,
        hspace=0.85, wspace=0.25,
        left=0.045, right=0.995, top=0.93, bottom=0.05,
    )

    # Column header strip
    for c, seg_def in enumerate(SEGMENT_COLUMNS):
        ax_h = fig.add_subplot(gs[0, c])
        ax_h.text(0.5, 0.4, column_title(*seg_def),
                  ha='center', va='center', fontsize=11, fontweight='bold')
        ax_h.axis('off')

    # Subplot grid
    axes_grid = [[None] * ncols for _ in range(nrows_full)]
    for r, row_def in enumerate(ROW_DEFS):
        for c, seg_def in enumerate(SEGMENT_COLUMNS):
            ax = fig.add_subplot(gs[r + 1, c])
            axes_grid[r][c] = ax
            fill_subplot(ax, df_raw, t, row_def, seg_def)

            # Y-axis label only on the leftmost column of each row
            if c == 0:
                ax.set_ylabel(row_def[0], fontsize=8, fontweight='bold')

            # Time tick labels are shown on every row; only the bottom row gets
            # an explicit 'Time (s)' axis label to avoid clutter.
            if r == nrows_full - 1:
                ax.set_xlabel('Time (s)', fontsize=12)

    # Title
    fig.suptitle(
        f'Glove sensor overview - all segments\n'
        f'{trial_name}    ·    {filt_desc}    ·    fs = {FS_HZ:g} Hz    ·    scale: {scale_desc}',
        fontsize=22, fontweight='bold', y=0.985,
    )
    pdf.savefig(fig, orientation='landscape')
    plt.close(fig)

    # ---------- Pages 2..N: one page per row, larger plots -------------------
    for row_def in ROW_DEFS:
        fig = plt.figure(figsize=PDF_FIG_SIZE)
        gs = gridspec.GridSpec(
            2, ncols,
            height_ratios=[0.10, 1],
            hspace=0.25, wspace=0.25,
            left=0.045, right=0.995, top=0.90, bottom=0.10,
        )

        for c, seg_def in enumerate(SEGMENT_COLUMNS):
            ax_h = fig.add_subplot(gs[0, c])
            ax_h.text(0.5, 0.4, column_title(*seg_def),
                      ha='center', va='center', fontsize=12, fontweight='bold')
            ax_h.axis('off')

        for c, seg_def in enumerate(SEGMENT_COLUMNS):
            ax = fig.add_subplot(gs[1, c])
            fill_subplot(ax, df_raw, t, row_def, seg_def)
            ax.set_xlabel('Time (s)', fontsize=8)
            if c == 0:
                ax.set_ylabel(row_def[0], fontsize=10, fontweight='bold')

            # ---- High-resolution y-axis tickers (per-row pages) ----
            # ~4x the default tick density, with minor ticks in between.
            # Skip cells that were rendered as 'n/a' (their spines are off).
            if any(s.get_visible() for s in ax.spines.values()):
                ax.yaxis.set_major_locator(MaxNLocator(nbins=24, steps=[1, 2, 2.5, 5, 10]))
                ax.yaxis.set_minor_locator(AutoMinorLocator(2))
                ax.tick_params(axis='y', which='major', labelsize=9)
                ax.tick_params(axis='y', which='minor', length=3)
                ax.grid(True, axis='y', which='major', alpha=0.35)
                ax.grid(True, axis='y', which='minor', alpha=0.15, linestyle=':')

        fig.suptitle(
            f'{row_def[0]}\n'
            f'{trial_name}    ·    {filt_desc}    ·    fs = {FS_HZ:g} Hz    ·    scale: {scale_desc}',
            fontsize=22, fontweight='bold', y=0.985,
        )
        pdf.savefig(fig, orientation='landscape')
        plt.close(fig)

    # PDF metadata
    info = pdf.infodict()
    info['Title']   = f'Glove signal overview - {trial_name}'
    info['Subject'] = f'Filter: {filt_desc}  |  Scale: {scale_desc}'
    info['Author']  = 'glove_data_visualiser_v3'

print(f'PDF written: {out_pdf}')
print(f'  Pages: {1 + nrows_full}  ·  Columns/page: {ncols}  ·  Filter: {filt_desc}')

PDF written: /home/jestin/ThesisRepo/ML/NewReports/DataGraphs/glove_data_8_Jestin_['Double', 'Pistol', 'Recoil']_visualization_Butterworth LP  (order=4, fc=3.0 Hz, zero-phase)_raw.pdf
  Pages: 6  ·  Columns/page: 14  ·  Filter: Butterworth LP  (order=4, fc=3.0 Hz, zero-phase)
